# Computing a social cost of carbon with dscim-cil

dscim is the Climate Impact Lab's library for computing the social cost
of carbon (SCC): it turns sector-level climate damages into damage
functions, applies them to FaIR climate projections, discounts, and
integrates to an SCC. dscim-cil drives that library from a YAML config.

Everything below runs on small generated inputs; no external data is
needed. The `run` extra must be installed
(`uv pip install ".[run]"`) so dscim itself is available.

## Inputs and config

Small versions of every input a run needs are generated here: an
economics zarr, reduced-damage zarrs, a GMST table, FaIR temperature
projections, and a pulse conversion file. The config below points at them and is saved as
`demo_data/demo.yml`.

In [ ]:
import pathlib
import sys

import yaml

repo = pathlib.Path.cwd().resolve()
if not (repo / "tests").exists():
    repo = repo.parent
sys.path.insert(0, str(repo / "tests"))
import fixture_factory

data = repo / "examples" / "demo_data"
data.mkdir(exist_ok=True)
config = fixture_factory.ssp_fixture_config(data)
config_path = data / "demo.yml"
config_path.write_text(yaml.safe_dump(config))
print(f"wrote {config_path}")

## The pipeline

`stages` explains the pipeline without touching any data: each stage,
what it reads and writes, and which dimensions it collapses.

In [ ]:
!dscim-cil stages

## Options

`options` lists every option dscim accepts. Each carries a status:
`supported`, `unsupported` (accepted by dscim but broken or restricted,
with the reason), `dead` (accepted and ignored), or `removed` (only on
other dscim branches).

In [ ]:
!dscim-cil options --status unsupported

`explain` gives the full record for one option, including why a
value is unsupported and where in dscim's source that behavior lives:

In [ ]:
!dscim-cil explain discounting_type constant_gwr

## Constraints

Some restrictions span options. `constraints` lists them with their
source citations; what it prints is what `validate` enforces.

In [ ]:
!dscim-cil constraints

## Validation

`validate` reports every problem at once.

In [ ]:
!dscim-cil validate {config_path}

## The plan

`plan` lays out the pipeline for this config as ordered steps, each
marked ready or blocked, with the command that produces every missing
input.

In [ ]:
!dscim-cil plan {config_path}

A dry run first prints the settings the sweep will use, marking
which values came from the config and which are dscim defaults (the
result-selecting values must always be explicit), then summarizes the
expanded runs:

In [ ]:
!dscim-cil run {config_path} --dry-run

## Running

One real run: fit the damage function on the generated damages, apply
it to the FaIR projections, discount, and write SCC files. On inputs
this small it takes a few seconds.

In [ ]:
!dscim-cil run {config_path}

## Results

Every run writes its artifacts plus a `*_run_metadata.yaml` recording
the resolved settings with their provenance, the dscim version and
commit that produced the result, and dependency versions.

In [ ]:
import xarray as xr

results = data / "results" / "labor" / "2020" / "unmasked"
stem = "adding_up_euler_ramsey_eta2.0_rho0.0001"
scc = xr.open_dataset(results / f"{stem}_scc.nc4")
print(scc)

In [ ]:
metadata = yaml.safe_load((results / f"{stem}_run_metadata.yaml").read_text())
print("dscim:", metadata["dscim_version"], "commit", metadata["dscim_commit"])
print("eta came from:", metadata["provenance"]["eta"])
print("ext_method came from:", metadata["provenance"]["ext_method"])